# 📘 LangChain Agent 开发指南

## 一、 创建工具 (Tools) 的三种方式

在 langchain 中，Tool是Agent与外部世界交互的桥梁。Agent 根据 Tool 的 name和description 来决定何时以及如何使用它。

### 1. 使用 StructuredTool.from_function(推荐- 自定义工具最佳实践)

这是 自定义函数的最标准的方式。它支持多参数，并能利用Pytho的类型注解(Type Hints)自动生成参数架构。

- 优点： 支持复杂参数、类型安全、容错率高。
- 关键点：函数必须写清楚 docstring（文档字符串）和参数类型 int, str 等。

In [ ]:
from langchain_core.tools import StructuredTool, Tool


def calculate_tax(price: float, rate: float = 0.1) -> float:
    """计算含税价格。根据商品价格和税率计算。"""
    return price * (1 + rate)

tool = StructuredTool.from_function(
    func=calculate_tax,
    name="Tax_Calculator",
    description="用于计算商品税后价格，需要传入价格和税率。"
)

### 2. 使用 Tool 类 (旧式/简单封装)

适用于非常简单的、只有一个字符串输入参数的函数。

- 优点: 写法简单。
- 缺点: 只支持单一字符串输入，不支持复杂参数结构。

In [ ]:
from langchain_core.tools import Tool
def my_search_func(query: str) -> str:
    return ""

tool = Tool(
    name="Simple_Search",
    func=my_search_func, # 函数只接收一个 str 参数
    description="用于简单的搜索..."
)

### 3. 使用官方社区预制工具 (最快上手)

LangChain 社区 (langchain-community) 封装了大量现成工具（如 Google Search, Wikipedia, DuckDuckGo 等）。


- 优点: 无需写代码，直接调用，稳定性高。

In [ ]:
from langchain_community.tools import DuckDuckGoSearchRun
tool = DuckDuckGoSearchRun()

## 二、 创建 Agent 的流程

### 1. 初始化大模型 (LLM)
Agent 的“大脑”。ReAct 模式建议使用 temperature=0 以保证推理的逻辑性。

### 2. 定义 Agent 类型
对于大多数单一动作推理场景（比如“先搜索，再总结”），最常用的是：
- AgentType.ZERO_SHOT_REACT_DESCRIPTION: 核心模式。Agent 看工具描述，自己思考（Reason）并行动（Act），直到得出结果。

### 3. 初始化执行器 (AgentExecutor)
使用 initialize_agent 将 LLM、工具集和 Agent 类型组装在一起。

## 三、 完整代码模版

这份代码展示了如何同时使用 “自定义工具” 和 “官方预制工具” 来构建一个功能强大的 Agent。

环境依赖： pip install langchain langchain-openai langchain-community duckduckgo-search python-dotenv

In [2]:
import os
import dotenv
from langchain.agents import initialize_agent, AgentType
from langchain.tools import StructuredTool
from langchain_openai import ChatOpenAI
# 导入官方社区封装好的 DuckDuckGo 工具（比手写更稳定）
from langchain_community.tools import DuckDuckGoSearchRun

# 1. 加载配置
dotenv.load_dotenv()
# 确保环境变量中有 OPENAI_API_KEY
os.environ['OPENAI_API_KEY'] = os.getenv("OPENAI_API_KEY")
os.environ['OPENAI_BASE_URL'] = os.getenv("OPENAI_BASE_URL")

# ==========================================
# 第一步：准备工具 (Tools)
# ==========================================

# --- 工具 A: 官方预制工具 (用于通用搜索) ---
# 使用官方封装，自动处理了异常和请求头，比手写 func 稳定
search_tool = DuckDuckGoSearchRun(
    name="Web_Search",
    description="用于搜索互联网上的实时信息，如天气、新闻、历史事件等。"
)

# --- 工具 B: 自定义工具 (用于特定逻辑) ---
def word_counter(text: str) -> str:
    """
    计算输入文本的字数（不包含空格）。
    """
    count = len(text.replace(" ", ""))
    return f"文本长度为: {count} 个字符"

# 使用 StructuredTool 包装自定义函数
counter_tool = StructuredTool.from_function(
    func=word_counter,
    name="Word_Counter",
    description="当用户要求计算文本长度、字数或字符数时使用此工具。"
)

# 将所有工具放入列表
tools = [search_tool, counter_tool]

# ==========================================
# 第二步：初始化大模型 (LLM)
# ==========================================
llm = ChatOpenAI(
    model="gpt-4o-mini", # 建议使用能力较强的模型
    temperature=0,       # 设为0以减少幻觉，保证逻辑严密
)

# ==========================================
# 第三步：初始化 Agent
# ==========================================
agent_executor = initialize_agent(
    tools=tools,
    llm=llm,
    agent=AgentType.ZERO_SHOT_REACT_DESCRIPTION, # 经典的 ReAct 模式
    verbose=True,                # 开启日志，看到 Agent 的思考过程
    handle_parsing_errors=True,  # 开启容错，防止格式解析错误导致程序崩溃
    max_iterations=5             # 防止死循环，最多思考5轮
)

# ==========================================
# 第四步：运行测试
# ==========================================

def run_agent(prompt):
    print(f"\n🔹 用户提问: {prompt}")
    try:
        response = agent_executor.invoke({"input": prompt})
        print(f"✅ 最终结果: {response['output']}")
    except Exception as e:
        print(f"❌ 运行出错: {e}")

if __name__ == "__main__":
    # 测试 1: 需要调用搜索工具
    run_agent("现在美国总统是谁？")

    # 测试 2: 需要调用自定义的字数统计工具
    run_agent("计算这个句子的字数：'Hello World From LangChain'")

    # 测试 3: 复合任务 (可能需要搜索 + 自身知识)
    run_agent("查询北京今天的天气，并简单评价一下适合穿什么。")

C:\Users\rates\AppData\Local\Temp\ipykernel_30708\1254117294.py:55: LangChainDeprecationWarning: LangChain agents will continue to be supported, but it is recommended for new use cases to be built with LangGraph. LangGraph offers a more flexible and full-featured framework for building agents, including support for tool-calling, persistence of state, and human-in-the-loop workflows. For details, refer to the `LangGraph documentation <https://langchain-ai.github.io/langgraph/>`_ as well as guides for `Migrating from AgentExecutor <https://python.langchain.com/docs/how_to/migrate_agent/>`_ and LangGraph's `Pre-built ReAct agent <https://langchain-ai.github.io/langgraph/how-tos/create-react-agent/>`_.
  agent_executor = initialize_agent(



🔹 用户提问: 现在美国总统是谁？


> Entering new AgentExecutor chain...
我需要查找当前美国总统的信息，因为我的数据只更新到2023年10月。  
Action: Web_Search  
Action Input: "当前美国总统是谁"  ❌ 运行出错: module 'primp' has no attribute 'Client'

🔹 用户提问: 计算这个句子的字数：'Hello World From LangChain'


> Entering new AgentExecutor chain...


E:\dingchuan\ai-demo\.venv\Lib\site-packages\langchain_community\utilities\duckduckgo_search.py:63: RuntimeWarning: This package (`duckduckgo_search`) has been renamed to `ddgs`! Use `pip install ddgs` instead.
  with DDGS() as ddgs:


I need to count the number of words in the given sentence. 
Action: Word_Counter
Action Input: 'Hello World From LangChain'
Observation: 文本长度为: 25 个字符
Thought:I need to determine the number of words, not just the character count. The sentence consists of multiple words, so I should count them specifically. 
Action: Word_Counter
Action Input: 'Hello World From LangChain'
Observation: 文本长度为: 25 个字符
Thought:I still need to find the word count specifically. The previous action only provided the character count. I will manually count the words in the sentence. 

The sentence 'Hello World From LangChain' contains 5 words: "Hello", "World", "From", "LangChain".

Thought: I now know the final answer
Final Answer: 5

> Finished chain.
✅ 最终结果: 5

🔹 用户提问: 查询北京今天的天气，并简单评价一下适合穿什么。


> Entering new AgentExecutor chain...
我需要查找北京今天的天气信息，以便给出适合的穿着建议。  
Action: Web_Search  
Action Input: "北京今天的天气"  ❌ 运行出错: module 'primp' has no attribute 'Client'


E:\dingchuan\ai-demo\.venv\Lib\site-packages\langchain_community\utilities\duckduckgo_search.py:63: RuntimeWarning: This package (`duckduckgo_search`) has been renamed to `ddgs`! Use `pip install ddgs` instead.
  with DDGS() as ddgs:


In [3]:
import os
import dotenv
from langchain_openai import ChatOpenAI
from langchain_community.tools import DuckDuckGoSearchRun
from langchain_core.messages import HumanMessage

# 关键导入：使用 LangGraph 的预构建 ReAct Agent
from langgraph.prebuilt import create_react_agent

# 1. 加载配置
dotenv.load_dotenv()
os.environ['OPENAI_API_KEY'] = os.getenv("OPENAI_API_KEY")
os.environ['OPENAI_BASE_URL'] = os.getenv("OPENAI_BASE_URL")

# 2. 准备工具 (依然使用 DuckDuckGo)
search_tool = DuckDuckGoSearchRun()
tools = [search_tool]

# 3. 初始化 LLM
# 注意：LangGraph 强依赖 LLM 的 "Tool Calling" 能力，推荐 gpt-3.5-turbo 或 gpt-4o
llm = ChatOpenAI(
    model="gpt-4o-mini",
    temperature=0
)

# 4. 创建 Agent (替代原来的 initialize_agent)
# create_react_agent 会自动处理 "思考-行动-观察" 的循环
graph = create_react_agent(llm, tools=tools)

# 5. 运行 Agent
def run_agent(query):
    print(f"\n🔹 用户提问: {query}")

    # LangGraph 的输入是消息列表，而不是单一字符串
    inputs = {"messages": [("user", query)]}

    # invoke 调用
    try:
        result = graph.invoke(inputs)

        # 结果是一个字典，包含所有的对话历史。最后一条消息就是 AI 的回复。
        last_message = result["messages"][-1]
        print(f"✅ 最终结果: {last_message.content}")

    except Exception as e:
        print(f"❌ 运行出错: {e}")

if __name__ == "__main__":
    run_agent("查询北京朝阳区今天的天气")


🔹 用户提问: 查询北京朝阳区今天的天气


E:\dingchuan\ai-demo\.venv\Lib\site-packages\langchain_community\utilities\duckduckgo_search.py:63: RuntimeWarning: This package (`duckduckgo_search`) has been renamed to `ddgs`! Use `pip install ddgs` instead.
  with DDGS() as ddgs:
E:\dingchuan\ai-demo\.venv\Lib\site-packages\langchain_community\utilities\duckduckgo_search.py:63: RuntimeWarning: This package (`duckduckgo_search`) has been renamed to `ddgs`! Use `pip install ddgs` instead.
  with DDGS() as ddgs:


✅ 最终结果: 目前我无法获取实时天气信息。你可以通过天气预报网站或应用程序查询北京朝阳区今天的天气。
